# Datathon 2026 — ÇOKLU Türkçe transformer (5-fold OOF meta-feature'lar)

BERTurk-base tek modeldi (+0.18 wOOF). Şimdi **2-3 FARKLI** Türkçe transformer fine-tune → her biri
AYRI meta-feature. Farklı mimariler farklı hata yapar → ensemble'da gerçek çeşitlilik (tek modelden fazla katkı).

Modeller (hepsi public/ungated):
- `dbmdz/bert-base-turkish-128k-cased` — büyük sözlük (128k), morfoloji daha iyi token'lanır
- `dbmdz/electra-base-turkish-cased-discriminator` — farklı ön-eğitim (replaced-token-detection), BERT'ten farklı hata
- `xlm-roberta-base` — çok-dilli, farklı tokenizer/mimari (opsiyonel; OOM olursa BATCH düşür veya kapat)

## ⚠️ GPU KURULUMU — SIRAYLA:
1. **Settings → Accelerator → GPU T4 ×2**
2. **Settings → Internet → ON** (ağırlıklar HF'den iner)
3. **Session RESTART** (accelerator değişince ŞART)
4. İlk çıktı `GPU: Tesla T4` olmalı.

## Çıktı (Run sonrası /kaggle/working'den indir, bana yükle):
Her model için `oof_<tag>_train.npy` + `<tag>_test.npy` (tag: bert128k, electra, xlmr).
Her model BİTER BİTMEZ kaydedilir → biri çökerse öncekiler durur.
Sonda: her modelin **BERTurk-base ile + birbiriyle korelasyonu** (düşük = gerçek çeşitlilik).

İsteğe bağlı: BERTurk-base OOF'unu (`oof_berturk_train.npy`) Add Input ile eklersen korelasyon matrisine girer.

In [ ]:
# ===================== ÇOKLU Türkçe transformer — 5-fold OOF =====================
import os, glob, random, numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), \
    "GPU YOK! Settings->Accelerator->GPU T4 -> Internet ON -> session RESTART -> tekrar."
print("GPU:", torch.cuda.get_device_name(0))

SEED, N_SPLITS = 42, 5
MAX_LEN, LR, MIN_EPOCHS, MAX_EPOCHS, BATCH, PATIENCE = 256, 2e-5, 2, 3, 16, 1
ID, TARGET, TEXT, YEAR = 'student_id', 'career_success_score', 'mentor_feedback_text', 'application_year'

# (tag, hf_model_id, batch_override or None). xlmr daha büyük -> gerekirse BATCH düşür/kapat.
MODELS = [
    ('bert128k', 'dbmdz/bert-base-turkish-128k-cased', None),
    ('electra',  'dbmdz/electra-base-turkish-cased-discriminator', None),
    ('xlmr',     'xlm-roberta-base', None),   # OOM olursa override=8 yap veya bu satırı sil
]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

# ---------- Veri + fold'lar (Faz 1-7 ile BİREBİR) ----------
base = [p for p in glob.glob('/kaggle/input/*') if os.path.exists(f'{p}/train.csv')]
if not base:
    hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    assert hits, "train.csv yok — Add Input -> Competitions -> Datathon 2026"
    base = [os.path.dirname(hits[0])]
base = base[0]
train = pd.read_csv(f'{base}/train.csv'); test = pd.read_csv(f'{base}/test_x.csv')
y = train[TARGET].values.astype('float32')
texts_tr = train[TEXT].fillna('').tolist(); texts_te = test[TEXT].fillna('').tolist()

from sklearn.model_selection import StratifiedKFold
tbin = pd.qcut(train[TARGET].values, 10, labels=False, duplicates='drop')
strat = train[YEAR].astype(str) + '_' + pd.Series(tbin).astype(str)
folds = list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(train, strat))
tr_prop = train[YEAR].value_counts(normalize=True); te_prop = test[YEAR].value_counts(normalize=True)
w = np.nan_to_num(train[YEAR].map(lambda yr: te_prop.get(yr,0.0)/tr_prop.get(yr,np.nan)).values)
def wmse(p): return float(np.sum(w*(y-p)**2)/np.sum(w))
dev = 'cuda'

from transformers import AutoTokenizer, AutoModelForSequenceClassification

def predict(model, ids, am, bs=64):
    model.eval(); out = []
    with torch.no_grad(), torch.cuda.amp.autocast():
        for s in range(0, len(ids), bs):
            o = model(input_ids=ids[s:s+bs].to(dev), attention_mask=am[s:s+bs].to(dev)).logits.squeeze(-1)
            out.append(o.float().cpu().numpy())
    return np.concatenate(out)

def run_model(tag, model_id, batch):
    print(f"\n{'='*60}\n[{tag}] {model_id}  (BATCH={batch})\n{'='*60}")
    tok = AutoTokenizer.from_pretrained(model_id)
    def enc_all(texts): return tok(texts, truncation=True, max_length=MAX_LEN, padding='max_length', return_tensors='pt')
    E_tr = enc_all(texts_tr); E_te = enc_all(texts_te)
    ids_tr, am_tr = E_tr['input_ids'], E_tr['attention_mask']
    ids_te, am_te = E_te['input_ids'], E_te['attention_mask']
    oof = np.zeros(len(train)); test_pred = np.zeros(len(test))
    for fi, (tr, va) in enumerate(folds):
        mu, sd = float(y[tr].mean()), float(y[tr].std())
        ytr = torch.tensor((y[tr]-mu)/sd, dtype=torch.float32)
        model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=1).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=LR); scaler = torch.cuda.amp.GradScaler()
        best_mse, best_pred, best_state, best_ep, no_improve = 1e9, None, None, -1, 0
        for ep in range(MAX_EPOCHS):
            model.train(); perm = torch.randperm(len(tr))
            for s in range(0, len(tr), batch):
                idx = perm[s:s+batch]
                gi = ids_tr[tr][idx].to(dev); ga = am_tr[tr][idx].to(dev); yb = ytr[idx].to(dev)
                opt.zero_grad()
                with torch.cuda.amp.autocast():
                    pr = model(input_ids=gi, attention_mask=ga).logits.squeeze(-1)
                    loss = ((pr - yb)**2).mean()
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            vp = predict(model, ids_tr[va], am_tr[va]) * sd + mu
            vmse = float(np.mean((y[va]-vp)**2))
            if vmse < best_mse - 1e-4:
                best_mse, best_pred, best_ep = vmse, vp.copy(), ep
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            print(f"  fold{fi} ep{ep} valMSE={vmse:.3f} (best ep{best_ep}={best_mse:.3f})")
            if ep >= MIN_EPOCHS and no_improve >= PATIENCE:
                print(f"    -> early stop"); break
        oof[va] = np.clip(best_pred, 0, 100)
        model.load_state_dict(best_state)
        test_pred += np.clip(predict(model, ids_te, am_te) * sd + mu, 0, 100) / N_SPLITS
        del model; torch.cuda.empty_cache()
    plain, weighted = float(np.mean((y-oof)**2)), wmse(oof)
    by_year = pd.DataFrame({'year': train[YEAR], 'e2': (y-oof)**2}).groupby('year')['e2'].mean()
    print(f"[{tag}] OOF: düz={plain:.4f} | ağırlıklı={weighted:.4f}  (BERTurk-base: düz~146.7)")
    print(by_year.round(3).to_string())
    # ANINDA KAYDET (sonraki model çökse de bu durur)
    np.save(f'/kaggle/working/oof_{tag}_train.npy', oof.astype('float32'))
    np.save(f'/kaggle/working/{tag}_test.npy', test_pred.astype('float32'))
    print(f"[{tag}] kaydedildi -> oof_{tag}_train.npy + {tag}_test.npy")
    return oof

# ---------- Tüm modelleri koştur ----------
oofs = {}
for tag, mid, ov in MODELS:
    try:
        oofs[tag] = run_model(tag, mid, ov or BATCH)
    except Exception as e:
        print(f"[{tag}] HATA atlandı: {repr(e)[:300]}")
        torch.cuda.empty_cache()

# ---------- Çeşitlilik: korelasyon matrisi ----------
# BERTurk-base OOF'u input olarak eklendiyse dahil et (düşük korelasyon = gerçek çeşitlilik)
bt = glob.glob('/kaggle/input/**/oof_berturk_train.npy', recursive=True)
if bt: oofs['berturk'] = np.load(bt[0])
if len(oofs) >= 2:
    C = pd.DataFrame(oofs)
    print("\n================ OOF korelasyon (Pearson) ================")
    print(C.corr().round(3).to_string())
    print("\n(berturk ile <~0.95 = çeşitlilik gerçek, ensemble'da ek katkı bekle)")
print("\nBİTTİ — /kaggle/working'deki tüm oof_*_train.npy + *_test.npy dosyalarını indir, bana yükle.")
